# Module 2: Machine Learning for Regression (Car Price Prediction)

**Project Goal:** গাড়ির বিভিন্ন ফিচার (যেমন: ব্র্যান্ড, ইঞ্জিন পাওয়ার, বয়স ইত্যাদি) বিশ্লেষণ করে গাড়ির দাম (`msrp`) প্রেডিক্ট করার জন্য একটি **Linear Regression** মডেল তৈরি করা।

**সম্পূর্ণ প্রজেক্টের ফ্লো (Workflow):**
`Raw Data` $\rightarrow$ `Data Cleaning & EDA` $\rightarrow$ `Train / Val / Test Split` $\rightarrow$ `Feature Engineering & Matrix (X)` $\rightarrow$ `Model Training (w0, w)` $\rightarrow$ `Validation & Regularization (Best r)` $\rightarrow$ `Final Model & Test Evaluation`

# 2.2 — Data Preparation

এই ধাপে আমরা ডেটাসেট লোড করব এবং সব কলামের নাম ও স্ট্রিং ভ্যালুগুলোকে ছোট হাতের অক্ষর (lowercase) করে স্পেসের বদলে আন্ডারস্কোর (`_`) বসাব, যাতে পুরো ডেটাসেটটি একরকম (uniform) থাকে।

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ডেটাসেট লোড করা
df = pd.read_csv('data.csv')
print("Dataset Shape:", df.shape)
df.head()

In [ ]:
# ১. কলামের নামগুলো একরকম (uniform) করা
df.columns = df.columns.str.lower().str.replace(' ', '_')

# ২. স্ট্রিং (object) কলামগুলো খুঁজে বের করে সেগুলোর ভ্যালুও একরকম করা
strings = list(df.dtypes[df.dtypes == 'object'].index)

for col in strings:
    df[col] = df[col].str.lower().str.replace(' ', '_')

df.head()

---
# 2.3 — Exploratory Data Analysis (EDA)

ডেটাসেটের ভেতরে কী ধরনের ভ্যালু আছে, কতগুলো মিসিং ভ্যালু আছে এবং আমাদের টার্গেট ভ্যারিয়েবল `msrp` (গাড়ির দাম)-এর ডিস্ট্রিবিউশন কেমন তা দেখব।

In [ ]:
# প্রতিটি কলামের প্রথম ৫টি ইউনিক ভ্যালু এবং মোট ইউনিক ভ্যালুর সংখ্যা দেখা
for col in df.columns:
    print(col)
    print(df[col].unique()[:5])
    print("Unique count:", df[col].nunique())
    print('-' * 30)

In [ ]:
# কোন কোন কলামে Missing Values (NaN) আছে তা চেক করা
df.isnull().sum()[df.isnull().sum() > 0]

### Target Variable (`msrp`) বিশ্লেষণ এবং Log Transformation
গাড়ির দামের গ্রাফ প্লট করলে দেখা যায় এটি **Long-tail (Right-skewed)** ডিস্ট্রিবিউশন—অর্থাৎ বেশিরভাগ গাড়ির দাম কম, কিন্তু অল্প কিছু গাড়ির দাম অস্বাভাবিক বেশি (যেমন ১০-২০ লাখ ডলার)।

Linear Regression মডেলের ভালো পারফরম্যান্সের জন্য **Normal (Bell-shaped) Distribution** দরকার। তাই আমরা দামের ওপর `np.log1p()` অর্থাৎ $\log(1 + y)$ অ্যাপ্লাই করব। এতে মানগুলোর বিশাল পার্থক্য কমে আসবে এবং গ্রাফটি দেখতে ঘণ্টার মতো (Bell-shaped) হবে।

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# ১. সম্পূর্ণ MSRP ডিস্ট্রিবিউশন (Long-tail)
sns.histplot(df.msrp, bins=50, ax=axes[0])
axes[0].set_title("Original MSRP Distribution")

# ২. ১ লাখ ডলারের নিচের গাড়িগুলোর দাম
sns.histplot(df.msrp[df.msrp < 100000], bins=50, ax=axes[1])
axes[1].set_title("MSRP < $100,000")

# ৩. Log Transformed MSRP (Normal / Bell-shaped)
price_logs = np.log1p(df.msrp)
sns.histplot(price_logs, bins=50, ax=axes[2])
axes[2].set_title("Log(1 + MSRP) Distribution")

plt.tight_layout()
plt.show()

---
# 2.4 — Setting Up the Validation Framework

মডেল সঠিকভাবে পরীক্ষা করার জন্য ডেটাসেটকে প্রথমে শাফল (shuffle) করে ৩ ভাগে ভাগ করব:
* **Train (60%):** মডেলকে শেখানোর জন্য।
* **Validation (20%):** সেরা ফিচার ও হাইপারপ্যারামিটার ($r$) নির্বাচন করার জন্য।
* **Test (20%):** একদম শেষে চূড়ান্ত মডেলের পারফরম্যান্স যাচাই করার জন্য।

In [ ]:
# ১. সাইজ নির্ধারণ করা
n = len(df)
n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

# ২. ইন্ডেক্স শাফল করা (reproducibility-র জন্য seed=2 দেওয়া হলো)
np.random.seed(2)
idx = np.arange(n)
np.random.shuffle(idx)

df_shuffled = df.iloc[idx].reset_index(drop=True)

# ৩. ডেটাসেট ভাগ করা
df_train = df_shuffled.iloc[:n_train].copy()
df_val = df_shuffled.iloc[n_train:n_train + n_val].copy()
df_test = df_shuffled.iloc[n_train + n_val:].copy()

print(f"Train: {len(df_train)}, Validation: {len(df_val)}, Test: {len(df_test)}")

In [ ]:
# ৪. টার্গেট ভ্যারিয়েবল (y) আলাদা করা এবং Log Transform করা
y_train = np.log1p(df_train.msrp.values)
y_val = np.log1p(df_val.msrp.values)
y_test = np.log1p(df_test.msrp.values)

# ৫. Data Leakage রোধ করতে মূল ডেটাফ্রেম থেকে msrp কলাম মুছে ফেলা
del df_train['msrp']
del df_val['msrp']
del df_test['msrp']

---
# 2.5 & 2.6 — Linear Regression (Simple & Vector Form)

**Linear Regression-এর মূল ধারণা:**
একটি গাড়ির ফিচারগুলোকে ($x_1, x_2, ...$) তাদের নিজস্ব গুরুত্ব বা **Weight** ($w_1, w_2, ...$) দিয়ে গুণ করে, তার সাথে বেসলাইন দাম বা **Bias** ($w_0$) যোগ করলেই প্রেডিকশন পাওয়া যায়:

$$g(x_i) = w_0 + w_1 x_{i1} + w_2 x_{i2} + \dots + w_n x_{in} = w_0 + x_i^T w$$

**মনে রাখবে:** যেহেতু আমরা মডেলটিকে লগ-ট্রান্সফর্মড দামের ওপর চালাচ্ছি, তাই মডেলের আউটপুটও আসবে লগ স্কেলে। আসল ডলারে দাম পেতে শেষে `np.expm1()` (অর্থাৎ $e^y - 1$) ব্যবহার করতে হবে।

In [ ]:
# উদাহরণ হিসেবে ১০ নম্বর গাড়ির ৩টি ফিচার নেওয়া হলো: [engine_hp, city_mpg, popularity]
xi = [453, 11, 86]

# অনুমান করে নেওয়া Bias (w0) এবং Weights (w)
w0 = 7.17
w = [0.01, 0.04, 0.002]

def linear_regression_single(xi, w0, w):
    pred = w0
    for j in range(len(xi)):
        pred = pred + w[j] * xi[j]
    return pred

log_pred = linear_regression_single(xi, w0, w)
actual_price = np.expm1(log_pred)

print(f"Log Prediction: {log_pred:.3f} -> Predicted Price: ${actual_price:,.2f}")

### Vector / Matrix Form
সবগুলো গাড়ির প্রেডিকশন একসাথে বের করার জন্য আমরা লুপ না চালিয়ে সরাসরি **Matrix Multiplication (`.dot()`)** ব্যবহার করি: $y_{\text{pred}} = w_0 + X \cdot w$।

---
# 2.7 — Training a Linear Regression Model (Normal Equation)

**মডেল ট্রেইনিং (Fitting) মানে কী?**
ট্রেইনিং ডেটা ($X$ এবং $y$) ব্যবহার করে সবচেয়ে সেরা **$w_0$ (Bias)** এবং **$w$ (Weights)** খুঁজে বের করাই হলো মডেল ট্রেইনিং। Linear Regression-এ এটি সরাসরি একটি গাণিতিক সূত্রের (**Normal Equation**) মাধ্যমে বের করা যায়:

$$w = (X^T X)^{-1} X^T y$$

* এখানে $X^T X$-কে বলা হয় **Gram Matrix**।
* Bias ($w_0$) হিসাব করার জন্য $X$ ম্যাট্রিক্সের শুরুতে সব `1`-এর একটি অতিরিক্ত কলাম (`np.ones`) যোগ করে নেওয়া হয়। ফলে প্রাপ্ত ফলাফলের প্রথম সংখ্যাটি (`w_full[0]`) হয় $w_0$ এবং বাকিগুলো (`w_full[1:]`) হয় ফিচারগুলোর ওয়েট $w$।

In [ ]:
def train_linear_regression(X, y):
    # ১. Bias (w0)-এর জন্য শুরুতে 1-এর কলাম যোগ করা
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])

    # ২. Normal Equation: w = (X^T * X)^-1 * X^T * y
    XTX = X.T.dot(X)
    XTX_inv = np.linalg.inv(XTX)
    w_full = XTX_inv.dot(X.T).dot(y)

    # ৩. Bias এবং Feature Weights আলাদা করে রিটার্ন করা
    return w_full[0], w_full[1:]

---
# 2.8, 2.9 & 2.10 — Baseline Model, RMSE & Validation

প্রথমে আমরা শুধুমাত্র ৫টি সংখ্যার কলাম (Numerical features) নিয়ে একটি সাধারণ **Baseline Model** তৈরি করব।

### Root Mean Squared Error (RMSE)
মডেলের প্রেডিকশন ($g(x_i)$) আসল দামের ($y_i$) চেয়ে গড়ে কতটা ভুল করছে তা মাপার মাপকাঠি হলো **RMSE**।

$$\text{RMSE} = \sqrt{\frac{1}{m} \sum_{i=1}^{m} (g(x_i) - y_i)^2}$$

* **RMSE যত কম হবে, আমাদের মডেল তত ভালো।**
* **মনে রাখবে:** মডেল ট্রেইন করব `X_train, y_train` দিয়ে, কিন্তু RMSE সবসময় চেক করব অদেখা ভ্যালিডেশন ডেটার (`X_val, y_val`) ওপর।

In [ ]:
# ৫টি বেসিক নিউমেরিক্যাল ফিচার
base = ['engine_hp', 'engine_cylinders', 'highway_mpg', 'city_mpg', 'popularity']

def prepare_X_base(df):
    # মিসিং ভ্যালুগুলোকে 0 দিয়ে পূরণ করে NumPy array বানানো
    return df[base].fillna(0).values

def rmse(y_true, y_pred):
    error = y_true - y_pred
    mse = (error ** 2).mean()
    return np.sqrt(mse)

In [ ]:
# ১. Training ডেটা দিয়ে মডেল ট্রেইন করা
X_train = prepare_X_base(df_train)
w0, w = train_linear_regression(X_train, y_train)

# ২. Validation ডেটার ওপর প্রেডিকশন এবং RMSE বের করা
X_val = prepare_X_base(df_val)
y_pred = w0 + X_val.dot(w)

print("Baseline Validation RMSE:", round(rmse(y_val, y_pred), 4))

---
# 2.11 — Simple Feature Engineering

গাড়ির দাম নির্ধারণে গাড়িটি কত বছরের পুরনো (`age`) তা অত্যন্ত গুরুত্বপূর্ণ। যেহেতু ডেটাসেটে সর্বশেষ বছর `2017`, তাই আমরা `age = 2017 - year` সূত্র দিয়ে নতুন একটি ফিচার তৈরি করব।

**প্রোগ্রামিং টিপস:** ফাংশনের ভেতর ডেটাফ্রেমে নতুন কলাম যোগ করার আগে অবশ্যই `df = df.copy()` করে নেবে, যাতে মূল `df_train` বা `df_val` ডেটাফ্রেমটি পরিবর্তন হয়ে না যায়।

In [ ]:
def prepare_X_age(df):
    df = df.copy()
    features = base.copy()

    df['age'] = 2017 - df['year']
    features.append('age')

    return df[features].fillna(0).values

# নতুন ফিচারসহ মডেল ট্রেইন ও ভ্যালিডেশন
X_train = prepare_X_age(df_train)
w0, w = train_linear_regression(X_train, y_train)

X_val = prepare_X_age(df_val)
y_pred = w0 + X_val.dot(w)

print("Validation RMSE (with age):", round(rmse(y_val, y_pred), 4))

**ফলাফল:** শুধুমাত্র `age` ফিচারটি যোগ করার ফলে আমাদের RMSE `0.7617` থেকে এক লাফে কমে **`0.5172`**-তে নেমে এসেছে! নিচে গ্রাফের মাধ্যমে এই উন্নতিটি স্পষ্ট দেখা যাচ্ছে:

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(y_val, color='blue', label='Actual MSRP (y_val)', bins=50, alpha=0.5)
sns.histplot(y_pred, color='red', label='Predicted MSRP (y_pred)', bins=50, alpha=0.5)
plt.title("Actual vs Predicted Distribution (With Age Feature)")
plt.legend()
plt.show()

---
# 2.12 — Categorical Variables (One-Hot Encoding)

মেশিন লার্নিং মডেল টেক্সট কলাম (যেমন: `make`, `transmission_type`) সরাসরি বুঝতে পারে না। এছাড়া `number_of_doors` (2, 3, 4) সংখ্যায় লেখা থাকলেও এটি আসলে একটি ক্যাটাগরি।

**ম্যানুয়াল One-Hot Encoding প্রক্রিয়া:**
1. `number_of_doors`-এর জন্য ৩টি বাইনারি (`0/1`) কলাম বানাব: `num_doors_2`, `num_doors_3`, `num_doors_4`।
2. বাকি ৭টি ক্যাটাগরিক্যাল কলামের প্রতিটির সবচেয়ে জনপ্রিয় (Most frequent) **৫টি করে ক্যাটাগরি** নিয়ে একটি ডিকশনারি (`categories`) তৈরি করব এবং প্রতিটির জন্য আলাদা `0/1` কলাম বানাব।

In [ ]:
categorical_columns = [
    'make', 'engine_fuel_type', 'transmission_type',
    'driven_wheels', 'market_category', 'vehicle_size', 'vehicle_style'
]

# প্রতিটি ক্যাটাগরিক্যাল কলামের Top 5 ভ্যালুগুলো ডিকশনারিতে সংরক্ষণ করা
categories = {}
for c in categorical_columns:
    categories[c] = list(df_train[c].value_counts().head().index)

# সম্পূর্ণ Feature Matrix তৈরির চূড়ান্ত ফাংশন
def prepare_X(df):
    df = df.copy()
    features = base.copy()

    # ১. গাড়ির বয়স (Age)
    df['age'] = 2017 - df['year']
    features.append('age')

    # ২. দরজার সংখ্যা (Number of doors)
    for v in [2, 3, 4]:
        df[f'num_doors_{v}'] = (df['number_of_doors'] == v).astype(int)
        features.append(f'num_doors_{v}')

    # ৩. সব ক্যাটাগরিক্যাল কলামের One-Hot Encoding
    for c, values in categories.items():
        for v in values:
            df[f'{c}_{v}'] = (df[c] == v).astype(int)
            features.append(f'{c}_{v}')

    return df[features].fillna(0).values

In [ ]:
X_train = prepare_X(df_train)
w0, w = train_linear_regression(X_train, y_train)

X_val = prepare_X(df_val)
y_pred = w0 + X_val.dot(w)

print("Validation RMSE (all features, no reg):", round(rmse(y_val, y_pred), 2))
print("Bias (w0):", w0)

**সমস্যা (Numerical Instability):**
সবগুলো ক্যাটাগরি যোগ করার পর দেখা যাচ্ছে RMSE কমে যাওয়ার বদলে উল্টোবেড়ে **`134.18`** (বা তার বেশি) হয়ে গেছে এবং $w_0$-এর মান এসেছে বিশাল বড় সংখ্যা ($-5.65 \times 10^{16}$)! এর সমাধান হলো **Regularization**।

---
# 2.13 — Regularization (Ridge Regression)

### কেন ওয়েটগুলো এত বিশাল হয়ে গেল?
অনেকগুলো বাইনারি কলাম যোগ করার ফলে $X$ ম্যাট্রিক্সের কিছু কলাম অন্য কলামগুলোর লিনিয়ার কম্বিনেশন বা প্রায় ডুপ্লিকেট হয়ে যায়। ফলে Gram Matrix ($X^T X$) প্রায় **Singular** হয়ে পড়ে এবং এর Inverse $(X^T X)^{-1}$ বের করতে গেলে ভেতরের সংখ্যাগুলো অকল্পনীয় বড় হয়ে যায়।

### সমাধান কী?
Gram Matrix ($X^T X$)-এর প্রধান কর্ণ (diagonal) বরাবর একটি ছোট সংখ্যা $r$ যোগ করা হয়। একে **L2 Regularization (বা Ridge Regression)** বলে:

$$w = (X^T X + r I)^{-1} X^T y$$

এখানে $I$ হলো Identity Matrix (`np.eye`)। এর ফলে ম্যাট্রিক্সটি আর কখনোই Singular হয় না এবং ওয়েটগুলো পুরোপুরি নিয়ন্ত্রণে চলে আসে।

In [ ]:
def train_linear_regression_reg(X, y, r=0.001):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])

    # কর্ণ বরাবর r যোগ করা (Regularization)
    XTX = X.T.dot(X)
    XTX = XTX + r * np.eye(XTX.shape[0])

    XTX_inv = np.linalg.inv(XTX)
    w_full = XTX_inv.dot(X.T).dot(y)

    return w_full[0], w_full[1:]

# r = 0.01 দিয়ে পরীক্ষা করা
X_train = prepare_X(df_train)
w0, w = train_linear_regression_reg(X_train, y_train, r=0.01)

X_val = prepare_X(df_val)
y_pred = w0 + X_val.dot(w)

print("Validation RMSE (with Regularization r=0.01):", round(rmse(y_val, y_pred), 5))

---
# 2.14 — Tuning the Model (Finding Best `r`)

এখানে $r$ হলো একটি হাইপারপ্যারামিটার। আমরা বিভিন্ন $r$ ভ্যালু দিয়ে মডেল ট্রেইন করে ভ্যালিডেশন ডেটায় দেখব কোন $r$-এর জন্য RMSE সবচেয়ে কম আসে।

In [ ]:
X_train = prepare_X(df_train)
X_val = prepare_X(df_val)

for r in [0.0, 0.00001, 0.0001, 0.001, 0.01, 0.1, 1, 10]:
    w0, w = train_linear_regression_reg(X_train, y_train, r=r)
    y_pred = w0 + X_val.dot(w)
    score = rmse(y_val, y_pred)
    print(f"r = {r:<8} | w0 = {w0:>12.5f} | Validation RMSE = {score:.5f}")

**সিদ্ধান্ত:** দেখা যাচ্ছে **`r = 0.001`** এর জন্য ওয়েটগুলোও স্বাভাবিক থাকে এবং ভ্যালিডেশন RMSE সবচেয়ে কম (**`0.45652`**) আসে।

---
# 2.15 — Using the Model (Final Training & Testing)

আমাদের সেরা হাইপারপ্যারামিটার (`r = 0.001`) নির্বাচন করা শেষ। এখন চূড়ান্ত ধাপ:
1. `df_train` এবং `df_val` একসাথে যুক্ত করে **`df_full_train` (80% ডেটা)** তৈরি করব এবং চূড়ান্ত মডেল ট্রেইন করব।
2. সম্পূর্ণ অদেখা **`df_test` (20% ডেটা)** দিয়ে ফাইনাল RMSE বের করব।
3. টেস্ট সেট থেকে একটি নির্দিষ্ট গাড়ি নিয়ে তার আসল দামের সাথে মডেলের প্রেডিক্ট করা দাম মিলিয়ে দেখব।

In [ ]:
# ১. Train এবং Validation ডেটা একত্রিত করা
df_full_train = pd.concat([df_train, df_val]).reset_index(drop=True)
X_full_train = prepare_X(df_full_train)
y_full_train = np.concatenate([y_train, y_val])

# ২. সেরা r=0.001 দিয়ে ফাইনাল মডেল ট্রেইন করা
w0, w = train_linear_regression_reg(X_full_train, y_full_train, r=0.001)

# ৩. Test সেটে চূড়ান্ত মূল্যায়ন
X_test = prepare_X(df_test)
y_pred_test = w0 + X_test.dot(w)

print("Final Test RMSE:", round(rmse(y_test, y_pred_test), 5))

Validation RMSE (`0.4565`) এবং Final Test RMSE (`0.4518`) খুবই কাছাকাছি। অর্থাৎ আমাদের মডেলটি ওভারফিটিং (Overfitting) করেনি এবং নতুন ডেটায় চমৎকার কাজ করছে।

### একটি একক গাড়ির দাম প্রেডিক্ট করা (Single Car Prediction)

In [ ]:
# টেস্ট সেটের ২০ নম্বর গাড়িটির তথ্য একটি ডিকশনারি হিসেবে নেওয়া
car = df_test.iloc[20].to_dict()

# ১টি গাড়ির ডিকশনারি দিয়ে ১ রো-বিশিষ্ট DataFrame তৈরি করে ফিচার ম্যাট্রিক্স বানানো
df_small = pd.DataFrame([car])
X_small = prepare_X(df_small)

# লগ স্কেলে দাম প্রেডিক্ট করা
y_pred_single = w0 + X_small.dot(w)
y_pred_single = y_pred_single[0]

# np.expm1() দিয়ে লগ স্কেল থেকে আসল ডলারে রূপান্তর করা
predicted_price = np.expm1(y_pred_single)
actual_price = np.expm1(y_test[20])

print(f"Car: {car['year']} {car['make'].title()} {car['model'].title()}")
print(f"Predicted Price : ${predicted_price:,.2f}")
print(f"Actual Price    : ${actual_price:,.2f}")

---
# একনজরে পুরো প্রজেক্টের মূল ৩টি পার্থক্য (Quick Recap)

1. **Training (`train_linear_regression_reg`):** ইনপুট হিসেবে ফিচার ম্যাট্রিক্স `X` এবং টার্গেট `y` দেওয়া হয় $\rightarrow$ আউটপুট হিসেবে মডেল তার শেখা ওয়েট **`w0, w`** রিটার্ন করে।
2. **Prediction (`w0 + X.dot(w)`):** ইনপুট হিসেবে শুধু নতুন ডেটার ফিচার ম্যাট্রিক্স `X` এবং আগে শেখা `w0, w` ব্যবহার করা হয় $\rightarrow$ আউটপুট হিসেবে লগ-স্কেলের প্রেডিকশন **`y_pred`** পাওয়া যায়।
3. **Evaluation (`rmse(y_actual, y_pred)`):** আসল টার্গেট `y` এবং প্রেডিক্ট করা `y_pred` তুলনা করে মডেলের ভুলের পরিমাণ (**RMSE**) বের করা হয়।